🚀 4. Best practical pipeline (DO THIS)
Step 1: Build dataset

From your data:

for each query:
    positives = gold citations
    negatives = BM25 top-k - positives
Step 2: Train embedding model

Use:

bge-base-en-v1.5 OR e5-base

Train with:

MultipleNegativesRankingLoss
Step 3: Build retriever
Encode all documents
FAISS index
Step 4: Train reranker

Use:

cross-encoder
input: (query, doc)
Step 5: Inference
Query
 → embedding retrieval (top 100)
 → reranker (top 100 → top 10)
 → threshold filter

### Step 1A: Get full text of citations

In [2]:
import pandas as pd

court_considerations = pd.read_csv("../data/court_considerations.csv")
laws_de = pd.read_csv("../data/laws_de.csv")

In [3]:
court_considerations.head()

,citation,text
0,BGE 139 I 2 E. 1.12.2011,betr. Verweigerung der Beiladung seien gutzuhe...
1,BGE 139 I 2 E. 2,Eventualiter sei die Rückweisung an die Vorins...
2,BGE 139 I 2 E. 5.1,"In der Sache ist vorweg zu prüfen, ob der Ents..."
3,BGE 139 I 2 E. 5.2,Art. 34 Abs. 1 BV gewährleistet in allgemeiner...
4,BGE 139 I 2 E. 5.3,Im vorliegenden Fall geht es nicht um die Gült...


In [4]:
laws_de.head()

,citation,text,title
0,Art. 1 112,Die Einwohnergemeinde Bern tritt der Schweizer...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
1,Art. 2 112,Die Einwohnergemeinde Bern wird ferner der Sch...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
2,Art. 3 Abs. 1 112,1 Falls die Schweizerische Eidgenossenschaft z...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
3,Art. 3 Abs. 2 112,2 Durch Anlage des neuen Verwaltungsgebäudes a...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
4,Art. 4 Abs. 1 112,1 Die Einwohnergemeinde Bern übernimmt im fern...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...


In [5]:
print("total: ", len(court_considerations) + len(laws_de)) # ~2.6 million entries

total:  2652248


### Below, we concluded that court considerations contains duplicate records, so we are merging them

In [6]:
# total unique citations in the combined dataset
print("unique citations laws_de: ", laws_de["citation"].nunique(), "all laws_length: ", len(laws_de))
print("unique citations court considerations: ", court_considerations["citation"].nunique(), "all court_considerations_length: ", len(court_considerations))

unique citations laws_de:  175933 all laws_length:  175933
unique citations court considerations:  1985178 all court_considerations_length:  2476315


In [7]:
# in court considerations, getting the duplicate citations
duplicate_citations = court_considerations[court_considerations.duplicated(subset=["citation"], keep=False)]

In [8]:
# for each citation, get the number of duplicates
duplicate_citations_count = duplicate_citations.groupby("citation").size().reset_index(name="count")

In [9]:
duplicate_citations_count

,citation,count
0,10Y.1/2003 05.11.2003 E. 1,3
1,10Y.1/2003 05.11.2003 E. 2,3
2,10Y.1/2003 05.11.2003 E. 3,2
3,11Z_1/2025 E. 3,2
4,12T_1/2007 29.05.2007 E. 1,2
...,...,...
441126,U 99/05 08.11.2005 E. 3,2
441127,U 99/06 25.04.2007 E. 1,2
441128,U 99/06 25.04.2007 E. 2,2
441129,U 99/06 25.04.2007 E. 3,2


### Analyzing train.csv

In [10]:
# checking in train.csv for the duplicate citations from the court considerations dataset and see how many of them are present in the train.csv
train = pd.read_csv("../data/train.csv")
train_duplicate_citations = train[train["gold_citations"].isin(court_considerations["citation"])]

In [11]:
# getting gold citations of train and analyzing how many of them are present in the court considerations dataset and how many of them are present in the laws_de dataset
unique_citation_freq = train["gold_citations"].str.split(";").explode().value_counts()
print("total unique gold citations in train: ", unique_citation_freq.shape[0])



# getting the number of gold citations in train that are present in court considerations and laws_de
court_considerations_citations = set(court_considerations["citation"].unique())
laws_de_citations = set(laws_de["citation"].unique())
train_citations_in_court_considerations = unique_citation_freq[unique_citation_freq.index.isin(court_considerations_citations)]
train_citations_in_laws_de = unique_citation_freq[unique_citation_freq.index.isin(laws_de_citations)]

print("number of unique gold citations in train that are present in court considerations: ", train_citations_in_court_considerations.shape[0])
print("number of unique gold citations in train that are present in laws_de: ", train_citations_in_laws_de.shape[0])


# print total number of training data points
print("total number of training data points: ", len(train))

# total number of gold citations, all sum
print("total number of gold citations in train: ", unique_citation_freq.sum())

# getting frequency of gold citations in train that are present in court considerations and laws_de
print("frequency of gold citations in train that are present in court considerations: ", train_citations_in_court_considerations.sum())
print("frequency of gold citations in train that are present in laws_de: ", train_citations_in_laws_de.sum())

train_citations_not_in_court_considerations_or_laws_de = unique_citation_freq[~unique_citation_freq.index.isin(court_considerations_citations) & ~unique_citation_freq.index.isin(laws_de_citations)]


# now print total, not unique gold citations in train that are not present in court considerations and laws_de
print("total number of gold citations in train that are not present in court considerations and laws_de: ", train_citations_not_in_court_considerations_or_laws_de.sum())


### print
print("\nTHE CONCLUSION FROM THE ABOVE ANALYSIS IS THAT THERE ARE A SIGNIFICANT NUMBER OF GOLD CITATIONS IN THE TRAINING DATA THAT ARE NOT PRESENT IN THE COURT CONSIDERATIONS AND LAWS_DE DATASETS. THIS MEANS THAT THE MODEL WILL HAVE TO LEARN TO GENERALIZE TO UNSEEN CITATIONS, WHICH COULD BE CHALLENGING. IT ALSO HIGHLIGHTS THE IMPORTANCE OF HAVING A DIVERSE AND COMPREHENSIVE TRAINING DATASET TO ENSURE THAT THE MODEL CAN LEARN EFFECTIVELY.")
# print the citations that are present in train but not in court considerations and laws_de both at the same time

print("number of unique gold citations in train that are not present in court considerations and laws_de: ", train_citations_not_in_court_considerations_or_laws_de.shape[0])




total unique gold citations in train:  2695
number of unique gold citations in train that are present in court considerations:  52
number of unique gold citations in train that are present in laws_de:  1876
total number of training data points:  1139
total number of gold citations in train:  4659
frequency of gold citations in train that are present in court considerations:  57
frequency of gold citations in train that are present in laws_de:  3262
total number of gold citations in train that are not present in court considerations and laws_de:  1340

THE CONCLUSION FROM THE ABOVE ANALYSIS IS THAT THERE ARE A SIGNIFICANT NUMBER OF GOLD CITATIONS IN THE TRAINING DATA THAT ARE NOT PRESENT IN THE COURT CONSIDERATIONS AND LAWS_DE DATASETS. THIS MEANS THAT THE MODEL WILL HAVE TO LEARN TO GENERALIZE TO UNSEEN CITATIONS, WHICH COULD BE CHALLENGING. IT ALSO HIGHLIGHTS THE IMPORTANCE OF HAVING A DIVERSE AND COMPREHENSIVE TRAINING DATASET TO ENSURE THAT THE MODEL CAN LEARN EFFECTIVELY.
number of

In [13]:
print("total train length: ", len(train))
print("duplicate citations in train: ", train_duplicate_citations.__len__())

total train length:  1139
duplicate citations in train:  5


### Same analysis for val.csv

In [14]:
# checking in val.csv for the duplicate citations from the court considerations dataset and see how many of them are present in the val.csv
val = pd.read_csv("../data/val.csv")
val_duplicate_citations = val[val["gold_citations"].isin(court_considerations["citation"])]

# getting gold citations of val and analyzing how many of them are present in the court considerations dataset and how many of them are present in the laws_de dataset
unique_citation_freq = val["gold_citations"].str.split(";").explode().value_counts()
print("total unique gold citations in val: ", unique_citation_freq.shape[0])



# getting the number of gold citations in val that are present in court considerations and laws_de
court_considerations_citations = set(court_considerations["citation"].unique())
laws_de_citations = set(laws_de["citation"].unique())
val_citations_in_court_considerations = unique_citation_freq[unique_citation_freq.index.isin(court_considerations_citations)]
val_citations_in_laws_de = unique_citation_freq[unique_citation_freq.index.isin(laws_de_citations)]

print("number of unique gold citations in val that are present in court considerations: ", val_citations_in_court_considerations.shape[0])
print("number of unique gold citations in val that are present in laws_de: ", val_citations_in_laws_de.shape[0])


# print total number of val data points
print("total number of validation data points: ", len(val))

# total number of gold citations, all sum
print("total number of gold citations in val: ", unique_citation_freq.sum())

# getting frequency of gold citations in val that are present in court considerations and laws_de
print("frequency of gold citations in val that are present in court considerations: ", val_citations_in_court_considerations.sum())
print("frequency of gold citations in val that are present in laws_de: ", val_citations_in_laws_de.sum())

val_citations_not_in_court_considerations_or_laws_de = unique_citation_freq[~unique_citation_freq.index.isin(court_considerations_citations) & ~unique_citation_freq.index.isin(laws_de_citations)]

# now print total, not unique gold citations in val that are not present in court considerations and laws_de
print("total number of gold citations in val that are not present in court considerations and laws_de: ", val_citations_not_in_court_considerations_or_laws_de.sum())


### print
print("\nTHE CONCLUSION FROM THE ABOVE ANALYSIS IS THAT THERE ARE A SIGNIFICANT NUMBER OF GOLD CITATIONS IN THE TRAINING DATA THAT ARE NOT PRESENT IN THE COURT CONSIDERATIONS AND LAWS_DE DATASETS. THIS MEANS THAT THE MODEL WILL HAVE TO LEARN TO GENERALIZE TO UNSEEN CITATIONS, WHICH COULD BE CHALLENGING. IT ALSO HIGHLIGHTS THE IMPORTANCE OF HAVING A DIVERSE AND COMPREHENSIVE TRAINING DATASET TO ENSURE THAT THE MODEL CAN LEARN EFFECTIVELY.")
# print the citations that are present in val but not in court considerations and laws_de both at the same time
print("number of unique gold citations in val that are not present in court considerations and laws_de: ", val_citations_not_in_court_considerations_or_laws_de.shape[0])




total unique gold citations in val:  222
number of unique gold citations in val that are present in court considerations:  101
number of unique gold citations in val that are present in laws_de:  121
total number of validation data points:  10
total number of gold citations in val:  251
frequency of gold citations in val that are present in court considerations:  102
frequency of gold citations in val that are present in laws_de:  149
total number of gold citations in val that are not present in court considerations and laws_de:  0

THE CONCLUSION FROM THE ABOVE ANALYSIS IS THAT THERE ARE A SIGNIFICANT NUMBER OF GOLD CITATIONS IN THE TRAINING DATA THAT ARE NOT PRESENT IN THE COURT CONSIDERATIONS AND LAWS_DE DATASETS. THIS MEANS THAT THE MODEL WILL HAVE TO LEARN TO GENERALIZE TO UNSEEN CITATIONS, WHICH COULD BE CHALLENGING. IT ALSO HIGHLIGHTS THE IMPORTANCE OF HAVING A DIVERSE AND COMPREHENSIVE TRAINING DATASET TO ENSURE THAT THE MODEL CAN LEARN EFFECTIVELY.
number of unique gold citati

## Analyzing Court considerations "citations" nomenclature

In [15]:
for i in court_considerations[court_considerations["citation"] == "U 99/06 25.04.2007 E. 1"]["text"]:
    print(i)

Das Bundesgesetz über das Bundesgericht vom 17. Juni 2005 (BGG; SR 173.110) ist am 1. Januar 2007 in Kraft getreten (AS 2006 1205, 1243). Da der angefochtene Entscheid vorher ergangen ist, richtet sich das Verfahren noch nach dem Bundesgesetz über die Organisation der Bundesrechtspflege vom 16. Dezember 1943 (OG; Art. 132 Abs. 1 BGG; BGE 132 V 393 E. 1.2 S. 395).
Die Verwaltungsgerichtsbeschwerde wird abgewiesen.


## Analyzing Laws "citations" nomenclature

## Making the dataset

In [16]:
citation_to_text = {}

for _, row in laws_de.iterrows(): # laws_de
    citation_to_text[row["citation"]] = row["title"] + " | " + row["text"]

for _, row in court_considerations.iterrows(): # court_considerations
    if (row["citation"] in citation_to_text):
        citation_to_text[row["citation"]] += " | " + row["text"]
    else:
        citation_to_text[row["citation"]] = row["text"]

In [7]:
# creating negatives using BM25
from rank_bm25 import BM25Okapi

# your documents
corpus = []
doc_ids = []

for cit, text in citation_to_text.items(): # takes around 1-2 minutes to run
    corpus.append(text.lower().split())   # simple tokenization
    doc_ids.append(cit)

In [6]:
print("corpus length: ", len(corpus))
print("doc_ids length: ", len(doc_ids))

corpus length:  175933
doc_ids length:  175933


In [7]:
bm25 = BM25Okapi(corpus) # takes around 7 minutes to run

In [8]:
def get_bm25_negatives(query, gold_citations, top_k=50, num_neg=5):
    # tokenize query
    tokenized_query = query.lower().split()
    
    # get scores
    scores = bm25.get_scores(tokenized_query)

    print(len(scores))
    
    # sort docs by score (descending)
    ranked_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    
    negatives = []
    
    for idx in ranked_indices:
        cit = doc_ids[idx]
        
        if cit not in gold_citations:
            negatives.append(cit)
        
        if len(negatives) >= num_neg:
            break
    
    return negatives

### building training data with positives and negatives

In [ ]:
import json
import os

if not os.path.exists("../data/train_data.json"):
    train_data = []

    for i, row in train.iterrows():
        if i % 1 == 0:
            print(f"Processing row {i}/{len(train)}")

        query = row["query"]
        gold_citations = set(row["gold_citations"].split(";"))
        
        # POSITIVES
        for cit in gold_citations:
            if cit in citation_to_text:
                train_data.append({
                    "query": query,
                    "doc": citation_to_text[cit],
                    "label": 1
                })
        
        # NEGATIVES (BM25)
        negatives = get_bm25_negatives(query, gold_citations, top_k=50, num_neg=5)
        
        for neg in negatives:
            train_data.append({
                "query": query,
                "doc": citation_to_text[neg],
                "label": 0
            })

    with open("../data/train_data.json", "w") as f:
        json.dump(train_data, f)
else:
    with open("../data/train_data.json", "r") as f:
        train_data = json.load(f)

### training reranker on train_data

In [9]:
from sentence_transformers import InputExample

train_examples = []

for item in train_data:
    train_examples.append(InputExample(texts=[item["query"], item["doc"]], label=item["label"]))

In [10]:
from torch.utils.data import DataLoader
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)

In [18]:
from sentence_transformers import CrossEncoder

model = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    num_labels=1,
)

/Users/keshavsharmaog/Downloads/swiss-law-agentic-retrieval/env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from datasets import Dataset

In [11]:
model.fit(
    train_dataloader=train_dataloader,
    epochs=3,              # start with 1
    warmup_steps=100,
    show_progress_bar=True,
    
)

Token indices sequence length is longer than the specified maximum sequence length for this model (752 > 512). Running this sequence through the model will result in indexing errors
/Users/keshavsharmaog/Downloads/swiss-law-agentic-retrieval/env/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
500,0.589000
1000,0.278600


In [12]:
train_data[0]

{'query': "Die A AG betreibt seit den 1970er-Jahren auf der Parzelle Nr. yyy (Wohn- und Gewerbezone) ein Recyclingunternehmen. Das Unternehmen verarbeitet vorwiegend Metallschrott. Dieser wird im Aussenbereich des Betriebsgeländes zwischengelagert und dort von einem fest instal lierten Metallschredder für die Weiterverarbeitung zerkleinert. Im Jahr 2018 wurde der 1983 bewilligte Metallschredder umfassend modernisiert; nur wenige Teile der Gehäusekonstruktion wurden weiterverwendet. Aufgrund der Modernisierung stieg die jährliche Verarbeitungskapazität von 75'000 auf 84'000 Tonnen, und die durchschnittliche Betriebsleistung pro Tag dehnte sich von zehn auf zwölf Stunden aus. Im Herbst 2020 kauft B das Wohnhaus auf der Parzelle Nr. zzz (Wohn- und Gewerbezone), welches an das Grundstück Nr. yyy angrenzt. Nach kurzer Zeit fühlt er sich vom Lärm des Me tallschredders gestört, zumal dieser gelegentlich bis in die späten Abendstunden (23.00 Uhr) in Betrieb ist. Er beschwert sich deshalb bei d

In [14]:
train_data[2]

{'query': "Die A AG betreibt seit den 1970er-Jahren auf der Parzelle Nr. yyy (Wohn- und Gewerbezone) ein Recyclingunternehmen. Das Unternehmen verarbeitet vorwiegend Metallschrott. Dieser wird im Aussenbereich des Betriebsgeländes zwischengelagert und dort von einem fest instal lierten Metallschredder für die Weiterverarbeitung zerkleinert. Im Jahr 2018 wurde der 1983 bewilligte Metallschredder umfassend modernisiert; nur wenige Teile der Gehäusekonstruktion wurden weiterverwendet. Aufgrund der Modernisierung stieg die jährliche Verarbeitungskapazität von 75'000 auf 84'000 Tonnen, und die durchschnittliche Betriebsleistung pro Tag dehnte sich von zehn auf zwölf Stunden aus. Im Herbst 2020 kauft B das Wohnhaus auf der Parzelle Nr. zzz (Wohn- und Gewerbezone), welches an das Grundstück Nr. yyy angrenzt. Nach kurzer Zeit fühlt er sich vom Lärm des Me tallschredders gestört, zumal dieser gelegentlich bis in die späten Abendstunden (23.00 Uhr) in Betrieb ist. Er beschwert sich deshalb bei d

In [15]:
query = train_data[2]["query"]

docs = [
    train_data[2]["doc"], # the label is 0 for this doc, but we want to see if the model can predict it correctly
    train_data[0]["doc"], # the label is 1 for this doc, but we want to see if the model can predict it correctly
]

pairs = [(query, d) for d in docs]

scores = model.predict(pairs)

print(scores)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


[-8.263328   6.4586806]


In [22]:
model.save("../models/cross_encoder_msmarco_finetuned")

## run from here, load model instead of training every time

In [17]:
# load model
reranker_model = CrossEncoder("../models/cross_encoder_msmarco_finetuned", num_labels=1)

NameError: name 'CrossEncoder' is not defined

### building retriver

In [19]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("intfloat/multilingual-e5-base", device="mps")

In [21]:
import numpy as np

docs = []
doc_ids = []

for cit, text in citation_to_text.items():
    # add E5 prefix
    docs.append("passage: " + text)
    doc_ids.append(cit)

# encode documents
doc_embeddings = model.encode(
    docs,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True   # IMPORTANT for cosine similarity
)

Batches:   1%|          | 58/10996 [00:57<3:00:59,  1.01it/s]


KeyboardInterrupt: 

faiss index

In [ ]:
import faiss

dim = doc_embeddings.shape[1]

index = faiss.IndexFlatIP(dim)  # Inner Product (cosine)

index.add(doc_embeddings)

retriver

In [ ]:
def retrieve(query, top_k=100):
    query_embedding = model.encode(
        ["query: " + query],
        normalize_embeddings=True
    )
    
    scores, indices = index.search(query_embedding, top_k)
    
    results = []
    
    for idx in indices[0]:
        results.append(doc_ids[idx])
    
    return results

### pipeline with reranker

In [ ]:

def full_pipeline(query, top_k=100, final_k=5):
    
    # Step 1: retrieve
    candidates = retrieve(query, top_k)
    
    # Step 2: prepare pairs
    pairs = [
        (query, citation_to_text[cit])
        for cit in candidates
    ]
    
    # Step 3: rerank
    scores = reranker_model.predict(pairs)
    
    # Step 4: sort
    ranked = sorted(
        zip(candidates, scores),
        key=lambda x: x[1],
        reverse=True
    )
    
    # Step 5: pick top
    final = [cit for cit, _ in ranked[:final_k]]
    
    return final

In [ ]:
from modal_runner import ModalRunner

runner = ModalRunner()
